In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy

contexto. camadas. neurônios. janela de observação.   

**Integração dos indicadores: Média Móvel Simples (SMA – Simple Moving Average)média aritmética dos preços de fechamento de um ativo durante um determinado período.**

**Média Móvel Exponencial (EMA – Exponential Moving Average) A EMA dá mais peso aos dados recentes, tornando-a mais sensível a mudanças recentes nos preços**

**Índice de Força Relativa (RSI – Relative Strength Index) oscilador que mede a velocidade e a mudança dos movimentos de preço**

In [ ]:
#INDICADORES
def add_indicators(df, col, window=14):

    df[f'SMA_{window}'] = df[col].rolling(window).mean()
    df[f'EMA_{window}'] = df[col].ewm(span=window, adjust=False).mean()

    delta = df[col].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss
    df[f'RSI_{window}'] = 100 - (100/(1+rs))

    return df

In [ ]:
tickers = ["AAPL","MSFT","GOOGL","META","AMZN"]

dfs=[]

for t in tickers:

    df = yf.download(t,start="2015-01-01",end="2025-01-01")

    # remover pandemia
    df = df[(df.index.year != 2020) & (df.index.year != 2021)]
    df = df[['Close']].rename(columns={'Close':f'{t}_Close'})
    df = add_indicators(df,col=f'{t}_Close')
    dfs.append(df)

data = pd.concat(dfs,axis=1).dropna()

**Normalização para prepar os dados para a LSTM**

In [ ]:

X = data.values
y = data[[f"{t}_Close" for t in tickers]].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X) ### (data leakage) usa dados do futuro para normalizar o passado
y_scaled = scaler_y.fit_transform(y)


def create_sequences(X,y,seq_len):

    Xs=[]
    ys=[]

    for i in range(seq_len,len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])

    return np.array(Xs),np.array(ys)

seq_len = 90 ### Janela de observação

X_seq,y_seq = create_sequences(X_scaled,y_scaled,seq_len)

split = int(0.8*len(X_seq))

X_train,X_test = X_seq[:split],X_seq[split:]
y_train,y_test = y_seq[:split],y_seq[split:]

class TimeSeriesDataset(Dataset):
    def __init__(self,X,y):

        self.X = torch.tensor(X,dtype=torch.float32)
        self.y = torch.tensor(y,dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]


train_ds = TimeSeriesDataset(X_train,y_train)
test_ds = TimeSeriesDataset(X_test,y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

**Uma camada LSTM seguida de uma GRU para capturar padrões temporais que tenha múltiplas variáveis, com um trecho de GRU que é bom para evitar overfitting**

In [ ]:

class LSTM_GRU_Model(nn.Module):

    def __init__(self,input_size,output_size):

        super().__init__()

        self.lstm = nn.LSTM(input_size,128,batch_first=True)
        self.gru = nn.GRU(128,128,batch_first=True)
        self.fc = nn.Linear(128,output_size)


    def forward(self,x):
        out_lstm,_ = self.lstm(x)
        out_gru,_ = self.gru(out_lstm)
        out = self.fc(out_gru[:,-1,:])
        return out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTM_GRU_Model(X_train.shape[2],len(tickers)).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)

In [ ]:
#TREINAMENTO
best_val_loss=float("inf")
best_model_state=None

train_losses=[]
val_losses=[]

epochs = 90

for epoch in range(1,epochs+1):

    model.train()
    batch_losses=[]

    for xb,yb in train_loader:
        xb,yb = xb.to(device),yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds,yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    train_loss=np.mean(batch_losses)

    model.eval()
    val_batch_losses=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            xb,yb = xb.to(device),yb.to(device)
            preds=model(xb)
            val_loss=criterion(preds,yb)
            val_batch_losses.append(val_loss.item())

    val_loss=np.mean(val_batch_losses)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch}: Train={train_loss:.6f} Val={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())

model.load_state_dict(best_model_state)

In [ ]:
#AVALIAÇÃO DO MODELO
model.eval()
preds=[]
with torch.no_grad():
    for xb,yb in test_loader:
        xb=xb.to(device)
        out=model(xb)
        preds.extend(out.cpu().numpy())
preds=np.array(preds)
real_prices = scaler_y.inverse_transform(y_test)
pred_prices = scaler_y.inverse_transform(preds)

**o quão bem o modelo conseguiu aprender o comportamento histórico da ação**

In [ ]:
for i,ticker in enumerate(tickers):
    plt.figure(figsize=(10,5))
    plt.plot(real_prices[:,i],label="Real")
    plt.plot(pred_prices[:,i],label="Predicted")
    plt.title(f"Stock Prediction - {ticker}")
    plt.legend()
    plt.grid(True)
    plt.show()

**desempenho do modelo no conjunto de teste com:**

**Modelo MAE = média dos erros absolutos entre os valores reais e os valores previstos**

**Modelo RMSE = raiz da média dos erros quadráticos entre valores reais e previstos = penaliza mais as previsões erradas**

In [ ]:
mae = mean_absolute_error(real_prices, pred_prices)
rmse = np.sqrt(mean_squared_error(real_prices, pred_prices))

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

In [ ]:
for i, ticker in enumerate(tickers):
    mae = mean_absolute_error(real_prices[:, i], pred_prices[:, i])
    rmse = np.sqrt(mean_squared_error(real_prices[:, i], pred_prices[:, i]))

    print(f"\n{ticker}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")

**avaliação do processo de aprendizado**

**Train Loss → erro nos dados de treino**

**Validation Loss → erro nos dados de teste**

In [ ]:

plt.figure(figsize=(8,4))
plt.plot(train_losses,label="Train Loss")
plt.plot(val_losses,label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

**tendência esperada por ação no curto prazo (7 dias)**

In [ ]:
n_future = 7
last_sequence = X_scaled[-seq_len:]
future_preds=[]
current_input = torch.tensor(last_sequence,dtype=torch.float32).unsqueeze(0).to(device)

for _ in range(n_future):
    with torch.no_grad():

        next_pred = model(current_input)
        pred_vals = next_pred.cpu().numpy()[0]
        future_preds.append(pred_vals)
    next_step = current_input[0,-1,:].cpu().numpy()
    next_sequence = np.vstack([current_input[0,1:].cpu(),next_step])
    current_input = torch.tensor(next_sequence,dtype=torch.float32).unsqueeze(0).to(device)
future_preds = np.array(future_preds)
future_prices = scaler_y.inverse_transform(future_preds)

print("\n==============================")
print("7-DAY STOCK FORECAST")
print("==============================\n")

for i,ticker in enumerate(tickers):
    print(f"{ticker}")
    for d in range(n_future):
        print(f" Day +{d+1}: ${future_prices[d,i]:.2f}")
    print()